In [3]:
# Load libraries
library("anndata")
library("DESeq2")
library("reticulate")

# Use the path to your Python executable in the virtual environment
use_python("/lila/home/forsythb/.virtualenvs/r-reticulate/bin/")

# Look at where Python is located
py_config()

# Import scanpy
sc <- import("scanpy")

python:         /lila/home/forsythb/.virtualenvs/r-reticulate/bin/python
libpython:      /home/forsythb/anaconda3/lib/libpython3.8.so
pythonhome:     /lila/home/forsythb/.virtualenvs/r-reticulate:/lila/home/forsythb/.virtualenvs/r-reticulate
version:        3.8.18 (default, Sep 11 2023, 13:40:15)  [GCC 11.2.0]
numpy:          /lila/home/forsythb/.virtualenvs/r-reticulate/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by use_python() function

In [2]:
# Read in the adata
adata <- sc$read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/postprocess_adata.020624/adata.combined.postprocess.h5ad')

In [3]:
# Filter rows for primary tumor
primary_tumor <- adata[rownames(adata$obs) %in% '146P', ]
write.h5ad(primary_tumor, '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/degs/primary_tumor.h5ad')

# Filter rows for metastatic tumor
metastatic_tumor <- adata[rownames(adata$obs) %in% 'KG146Li', ]
write.h5ad(metastatic_tumor, '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/degs/metastatic_tumor.h5ad')

ERROR: Error in write.h5ad(primary_tumor, "/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/degs/primary_tumor.h5ad"): could not find function "write.h5ad"


In [3]:
adata$obs

,background_fraction,cell_probability,cell_size,droplet_efficiency,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,⋯,mito_frac,Patient,Tumor_Site,Culture_Media,ZFP_Expression,Replicate,Batch,Sample,phenograph,leiden
,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<fct>,<fct>,<fct>
KG146Li_BASE_shZFP36L2_4_GTCGAATTCGGACGTC-1,0.015236451,0.9999540,15092.094,1.0152237,4410,8.391857,13139,9.483416,22.83279,32.26273,⋯,0.06469290,146,Metastatic,BASE,ZFP_KD,2,2,146_M_BASE_ZFPKD_2,21,22
KG146Li_HISC_shCtrl_GGATCTAAGGTTATAG-1,0.014875664,0.9999546,8818.770,1.1403950,2946,7.988543,6701,8.810161,25.62304,34.20385,⋯,0.09819430,146,Metastatic,HISC,CTRL,1,3,146_M_HISC_CTRL_1,0,0
146P_dedifferentiation_shCtrl_CTCTCAGAGGTAGACC-1,0.045285905,0.9999546,16044.765,1.2350457,3376,8.124743,16212,9.693569,64.03282,67.03676,⋯,0.13058228,146,Primary,Dedifferentiated,CTRL,1,8,146_P_Dediff_CTRL_1,16,17
146P_HISC_shZFP36L2_4_GTGCAGCAGAGAGGGC-1,0.045060653,0.9999998,11297.001,0.7787340,2657,7.885329,7163,8.876824,32.20718,43.13835,⋯,0.01884685,146,Primary,HISC,ZFP_KD,2,6,146_P_HISC_ZFPKD_2,15,16
146P_BASE_shZFP36L2_3_TAGAGTCAGAAAGTCT-1,0.021178186,0.9999994,8557.787,1.1305823,3076,8.031710,8042,8.992557,29.49515,38.98284,⋯,0.05931360,146,Primary,BASE,ZFP_KD,1,5,146_P_BASE_ZFPKD_1,4,29
146P_BASE_shCTRL_GGGTCACGTGATACTC-1,0.016049281,0.9999532,10327.634,1.3651079,3291,8.099251,12139,9.404261,40.48109,51.76703,⋯,0.04827416,146,Primary,BASE,CTRL,1,5,146_P_BASE_CTRL_1,9,7
KG146Li_BASE_shZFP36L2_4_TTGTTGTAGTCACTGT-1,0.016974458,0.9999232,15651.236,0.8909897,3869,8.261010,11735,9.370416,26.31444,36.42096,⋯,0.09569663,146,Metastatic,BASE,ZFP_KD,2,2,146_M_BASE_ZFPKD_2,21,22
146P_BASE_shCTRL_GGAGATGAGTTCCAGT-1,0.030841348,0.9999990,9991.460,0.9284167,2664,7.887959,7856,8.969160,34.82688,43.07536,⋯,0.06364562,146,Primary,BASE,CTRL,1,5,146_P_BASE_CTRL_1,9,7
146P_BASE_shZFP36L2_4_GAGTTTGCAAGACGGT-1,0.031681873,0.9999999,8755.726,1.0266340,2686,7.896181,7427,8.913012,26.53831,38.05036,⋯,0.05156860,146,Primary,BASE,ZFP_KD,2,5,146_P_BASE_ZFPKD_2,14,5


In [16]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
#count_matrix <- adata$X
#count_matrix <- as(count_matrix, "matrix")
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [17]:
count_matrix

0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
2,2,3,4,4,4,4,5,3,4,⋯,0,3,3,4,4,3,4,4,3,3
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,1,0,0,1
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,1,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,1,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [27]:
# Extract sample names from row names
sample_names <- rownames(adata$X)

# Assuming 'adata' is your AnnData object

# Extract sample names from row names
sample_names <- rownames(adata$X)

# Create a new column 'Tumor_Site' based on sample names
adata$obs$Tumor_Site <- ifelse(grepl("146P", sample_names), "Primary", 
                               ifelse(grepl("146Li", sample_names), "Metastatic", NA))

# Create a new column 'Culture_Media' based on sample names
adata$obs$Culture_Media <- ifelse(grepl("BASE", sample_names, ignore.case = TRUE), "BASE", 
                                  ifelse(grepl("HISC", sample_names, ignore.case = TRUE), "HISC",
                                  ifelse(grepl("dedifferentiation", sample_names, ignore.case = TRUE), "Dedifferentiated", "Unknown")))

# Create a new column 'ZFP_Expression' based on sample names
adata$obs$ZFP_Expression <- ifelse(grepl("CTRL", sample_names, ignore.case = TRUE), "CTRL", 
                                  ifelse(grepl("ZFP", sample_names, ignore.case = TRUE), "ZFPKD", "Unknown"))

In [28]:
adata$obs

,background_fraction,cell_probability,cell_size,droplet_efficiency,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,log10GenesPerUMI,Sample,original_total_counts,log10_original_total_counts,mito_frac,Tumor_Site,Culture_Media,ZFP_Expression
,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
146P_HISC_shCTRL_CCGGACACACCACATA-1,0.02229144,0.9999546,14050.673,1.3242252,4761,8.468423,16009,9.680969,40.98320,44.36879,49.00993,58.26722,0.8747335,146P_HISC_shCTRL,16009,4.204364,0.15010307,Primary,HISC,CTRL
146P_dedifferentiation_shZFP36L2_3_ACTTTGTTCGCTGTTC-1,0.05122740,0.9999504,14127.893,1.1506593,3471,8.152486,12909,9.465757,42.38903,48.98908,56.04617,67.08498,0.8612374,146P_dedifferentiation_shZFP36L2_3,12909,4.110893,0.04012704,Primary,Dedifferentiated,ZFPKD
KG146Li_BASE_shZFP36L2_3_GTGGTTAGTTGGGAAC-1,0.00761133,0.9999546,14619.754,2.0058451,5775,8.661467,26207,10.173820,36.40249,44.01496,51.05506,60.77765,0.8513347,KG146Li_BASE_shZFP36L2_3,26207,4.418417,0.05922082,Metastatic,BASE,ZFPKD
146P_dedifferentiation_shCtrl_GGGTCTGTCGATGCTA-1,0.03223666,0.9999546,15622.843,1.3579359,4567,8.426831,17502,9.770128,39.38978,45.82905,52.78825,62.98709,0.8624924,146P_dedifferentiation_shCtrl,17502,4.243088,0.08467604,Primary,Dedifferentiated,CTRL
146P_HISC_shZFP36L2_3_TGGAACTCAACCTAAC-1,0.01331819,0.9999546,15020.916,1.4536747,5019,8.521185,19114,9.858229,30.37564,39.54693,47.44690,58.25050,0.8643572,146P_HISC_shZFP36L2_3,19114,4.281352,0.06581563,Primary,HISC,ZFPKD
146Li_dedifferentiation_shZFP36L2_4_TCCCATGAGTGACCTT-1,0.03227444,0.9999546,16821.025,1.1827675,3116,8.044626,17151,9.749870,53.60037,61.56492,69.27293,78.52603,0.8250729,146Li_dedifferentiation_shZFP36L2_4,17151,4.234289,0.05655647,Metastatic,Dedifferentiated,ZFPKD
146P_dedifferentiation_shZFP36L2_3_GGCGTCAGTCCAGCGT-1,0.07923496,0.9999856,14131.530,0.8634863,2746,7.918265,9436,9.152393,46.47096,53.69860,60.61891,71.13184,0.8651280,146P_dedifferentiation_shZFP36L2_3,9436,3.974788,0.03179313,Primary,Dedifferentiated,ZFPKD
146P_BASE_shZFP36L2_3_TCTTTGAGTGCAGGAT-1,0.03844221,0.9999993,9622.292,0.9532719,2457,7.807103,7654,8.943114,36.19023,46.98197,57.10739,69.75438,0.8729409,146P_BASE_shZFP36L2_3,7654,3.883888,0.04115495,Primary,BASE,ZFPKD
146P_dedifferentiation_shZFP36L2_3_CATCAAGTCTCGTGAA-1,0.03585139,0.9999546,16797.027,1.1660888,4236,8.351611,16324,9.700453,45.05636,51.40897,57.77383,66.84024,0.8609317,146P_dedifferentiation_shZFP36L2_3,16324,4.212827,0.08999020,Primary,Dedifferentiated,ZFPKD


In [20]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)

# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + ZFP_Expression
)

converting counts to integer mode



In [21]:
# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names

# Set the reference levels
coldata$Tumor_Site<-relevel(coldata$Tumor_Site,ref="Primary")
coldata$Culture_Media<-relevel(coldata$Culture_Media,ref="BASE")
coldata$ZFP_Expression<-relevel(coldata$ZFP_Expression,ref="CTRL")

# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + ZFP_Expression 
)

converting counts to integer mode



In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

estimating size factors

estimating dispersions

gene-wise dispersion estimates



In [ ]:
# Specify the date
date <- '020224'

# Specify the directory path
directory_path <- paste("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/deseq/deseq.", date, sep="")

# Check if the directory exists, and create it if not
if (!dir.exists(directory_path)) {
  dir.create(directory_path, recursive = TRUE)
}

# Specify the file path where you want to save the results
result_file <- paste(directory_path, "/sc_zfpexp_results.csv", sep="")

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = paste(directory_path, "/sc_zfpexp_dds.rds", sep=""))

# Save the DESeq results
saveRDS(results(dds_1), file = paste(directory_path, "/sc_zfpexp_results.rds", sep=""))


### DEGs for Primary Tumor

In [4]:
# Read in the adata for primary tumor site
adata_primary = sc$read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/degs/primary_tumor.h5ad')

In [5]:
adata_primary

AnnData object with n_obs × n_vars = 30895 × 31586
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph', 'leiden'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'diffmap_evals', 'hvg', 'leiden', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'rank_genes_groups', 'umap', 'var_explained'
    obsm: 'X_diffmap', 'X_pca', 'X_umap', 'gene_expression_encoding'
    layers: 'w

In [6]:
# Get the counts matrix
counts = as.matrix(adata_primary$X)

# Normalize
ms <- rowSums(counts)
norm_df <- counts/ms * median(ms)

# Log transform
counts <- log(norm_df + 1)

# Add pseudocount
counts <- counts + 1

# Round since error that not all values are integers
counts <- round(counts)

# Transpose
counts <- t(counts)

# Assign row and column names to the counts matrix
rownames(counts) <- adata_primary$var_names
colnames(counts) <- adata_primary$obs_names

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 7.3 GiB”


In [55]:
# Extract count matrix from the raw layer
count_matrix <- as.matrix(adata_primary$X)
# Transpose the ocunt matrix
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)
# Add a pseudocount
count_matrix <- count_matrix + 1 
# Normalize the count matrix by library size
library_sizes <- colSums(count_matrix)
count_matrix <- count_matrix / library_sizes * mean(library_sizes)

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 7.3 GiB”


In [7]:
counts

,146P_dedifferentiation_shCtrl_CTCTCAGAGGTAGACC-1,146P_HISC_shZFP36L2_4_GTGCAGCAGAGAGGGC-1,146P_BASE_shZFP36L2_3_TAGAGTCAGAAAGTCT-1,146P_BASE_shCTRL_GGGTCACGTGATACTC-1,146P_BASE_shCTRL_GGAGATGAGTTCCAGT-1,146P_BASE_shZFP36L2_4_GAGTTTGCAAGACGGT-1,146P_BASE_shZFP36L2_3_CTAACCCTCACATCAG-1,146P_HISC_shCTRL_TGCAGATAGTCTACCA-1,146P_dedifferentiation_shZFP36L2_3_CACTTCGCATGTGGTT-1,146P_HISC_shCTRL_AAGAACAAGGACACTG-1,⋯,146P_dedifferentiation_shZFP36L2_4_TACCTCGTCGACTCCT-1,146P_BASE_shZFP36L2_4_AGGACGATCTCTCTTC-1,146P_dedifferentiation_shCtrl_AAGACTCGTGATATAG-1,146P_HISC_shZFP36L2_4_CCTCCTCCATATGGCT-1,146P_HISC_shZFP36L2_4_ATGTCTTCAGTCCCGA-1,146P_BASE_shCTRL_CCTCAGTCAGGTATGG-1,146P_HISC_shCTRL_CCTCACAGTTGGTGTT-1,146P_BASE_shZFP36L2_3_TCACGCTTCATAGGCT-1,146P_dedifferentiation_shZFP36L2_4_TCGGGTGCAAACGGCA-1,146P_BASE_shZFP36L2_4_ACTGCAATCAGTGTTG-1
A1BG,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A1BG-AS1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A1CF,2,1,1,2,1,1,1,1,1,1,⋯,1,1,2,2,1,1,1,2,1,1
A2M,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2M-AS1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2ML1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2ML1-AS1,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A2ML1-AS2,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A3GALT2,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
A4GALT,1,1,1,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1


In [ ]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = counts,
  colData = coldata,
  design = ~ Culture_Media + ZFP_Expression 
)

In [8]:
# Define the column data
coldata <- adata_primary$obs[,c("Culture_Media", "ZFP_Expression")]

# Convert column data to factors
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)

# Set the reference levels
coldata$Culture_Media<-relevel(coldata$Culture_Media,ref="BASE")
coldata$ZFP_Expression<-relevel(coldata$ZFP_Expression,ref="CTRL")

In [ ]:
# Add a pseudo-count of 1 to all counts
counts <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata_primary$var_names

# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Culture_Media + ZFP_Expression:Culture_Media 
)

In [ ]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = counts,
  colData = coldata
)

In [25]:
# Filter the DDS object
keep <- rowSums(counts(dds_1)) >= 10
dds_1 <- dds_1[keep,]

In [13]:
dds_1 <- estimateSizeFactors(dds_1)

In [15]:
dds_1 <- estimateDispersionsGeneEst(dds_1)

In [16]:
dispersions(dds_1) <- mcols(dds_1)$dispGeneEst

In [17]:
dds_1 <- nbinomWaldTest(dds_1)

In [21]:
dds_1$group <- factor(paste0(dds_1$Culture_Media, dds_1$ZFP_Expression))

In [23]:
dds_1$group

[1] DedifferentiatedCTRL   HISCZFP_KD             BASEZFP_KD            
    [4] BASECTRL               BASECTRL               BASEZFP_KD            
    [7] BASEZFP_KD             HISCCTRL               DedifferentiatedZFP_KD
   [10] HISCCTRL               HISCCTRL               BASEZFP_KD            
   [13] HISCZFP_KD             DedifferentiatedZFP_KD BASEZFP_KD            
   [16] DedifferentiatedZFP_KD BASEZFP_KD             HISCCTRL              
   [19] DedifferentiatedZFP_KD BASECTRL               DedifferentiatedZFP_KD
   [22] DedifferentiatedZFP_KD HISCZFP_KD             BASEZFP_KD            
   [25] DedifferentiatedZFP_KD BASEZFP_KD             BASECTRL              
   [28] BASECTRL               BASEZFP_KD             BASEZFP_KD            
   [31] BASECTRL               DedifferentiatedCTRL   BASEZFP_KD            
   [34] BASECTRL               DedifferentiatedCTRL   HISCZFP_KD            
   [37] DedifferentiatedZFP_KD HISCZFP_KD             DedifferentiatedZFP_KD
   [40] DedifferentiatedZFP_KD BASEZFP_KD             DedifferentiatedZFP_KD
   [43] BASEZFP_KD             HISCCTRL               HISCZFP_KD            
   [46] DedifferentiatedZFP_KD BASECTRL               BASECTRL              
   [49] DedifferentiatedCTRL   HISCCTRL               HISCZFP_KD            
   [52] BASEZFP_KD             HISCZFP_KD             DedifferentiatedCTRL  
   [55] DedifferentiatedCTRL   BASECTRL               HISCZFP_KD            
   [58] BASEZFP_KD             BASEZFP_KD             DedifferentiatedZFP_KD
   [61] DedifferentiatedZFP_KD BASECTRL               BASEZFP_KD            
   [64] DedifferentiatedCTRL   HISCCTRL               HISCZFP_KD            
   [67] DedifferentiatedCTRL   BASECTRL               DedifferentiatedZFP_KD
   [70] BASEZFP_KD             DedifferentiatedZFP_KD HISCZFP_KD            
   [73] HISCZFP_KD             BASEZFP_KD             BASECTRL              
   [76] HISCZFP_KD             DedifferentiatedCTRL   BASEZFP_KD            
   [79] BASEZFP_KD             HISCCTRL               HISCCTRL              
   [82] BASEZFP_KD             HISCZFP_KD             BASEZFP_KD            
   [85] HISCZFP_KD             BASECTRL               BASECTRL              
   [88] BASECTRL               BASEZFP_KD             BASEZFP_KD            
   [91] HISCCTRL               BASEZFP_KD             HISCCTRL              
   [94] DedifferentiatedZFP_KD BASECTRL               BASEZFP_KD            
   [97] DedifferentiatedZFP_KD BASECTRL               HISCZFP_KD            
  [100] DedifferentiatedCTRL   HISCZFP_KD             DedifferentiatedCTRL  
  [103] HISCZFP_KD             BASECTRL               BASEZFP_KD            
  [106] BASEZFP_KD             BASEZFP_KD             DedifferentiatedZFP_KD
  [109] DedifferentiatedZFP_KD HISCZFP_KD             HISCZFP_KD            
  [112] HISCCTRL               DedifferentiatedZFP_KD HISCCTRL              
  [115] DedifferentiatedZFP_KD HISCZFP_KD             DedifferentiatedZFP_KD
  [118] HISCZFP_KD             DedifferentiatedZFP_KD DedifferentiatedZFP_KD
  [121] DedifferentiatedZFP_KD BASEZFP_KD             DedifferentiatedZFP_KD
  [124] DedifferentiatedZFP_KD HISCZFP_KD             HISCZFP_KD            
  [127] BASEZFP_KD             HISCZFP_KD             DedifferentiatedZFP_KD
  [130] DedifferentiatedZFP_KD BASECTRL               DedifferentiatedZFP_KD
  [133] BASEZFP_KD             DedifferentiatedZFP_KD HISCZFP_KD            
  [136] BASECTRL               BASEZFP_KD             DedifferentiatedZFP_KD
  [139] HISCZFP_KD             HISCZFP_KD             DedifferentiatedCTRL  
  [142] HISCCTRL               BASECTRL               DedifferentiatedZFP_KD
  [145] DedifferentiatedZFP_KD DedifferentiatedCTRL   HISCZFP_KD            
  [148] BASECTRL               BASEZFP_KD             HISCCTRL              
  [151] DedifferentiatedCTRL   HISCZFP_KD             DedifferentiatedCTRL  
  [154] DedifferentiatedZFP_KD BASECTRL               DedifferentiatedZFP_KD


In [19]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

using pre-existing size factors

estimating dispersions

found already estimated dispersions, replacing these

gene-wise dispersion estimates



In [ ]:
# Specify the date
date <- '020224'

# Specify the directory path
directory_path <- paste("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/deseq/deseq.", date, sep="")

# Check if the directory exists, and create it if not
if (!dir.exists(directory_path)) {
  dir.create(directory_path, recursive = TRUE)
}

# Specify the file path where you want to save the results
result_file <- paste(directory_path, "/sc_zfpexp_results.csv", sep="")

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = paste(directory_path, "/sc_zfpexp_dds.rds", sep=""))

# Save the DESeq results
saveRDS(results(dds_1), file = paste(directory_path, "/sc_zfpexp_results.rds", sep=""))